###Step 1 : Import Libraries & API Keys

In [108]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown
import gradio as gr
import json
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API key is missing")

     

###Step 2: Set up Pushover

In [ ]:
#Step 2a -> set up account in browser pushover
#Step 2b-> Set up the app on your iphone/Android phone, Log into the same account
#step 2c-> In the browser create an "Application/API Token"
#Step 2d-> copy your user key and API token into the .env file
#like this but without the hastag symbols and with your own keys
#PUSHOVER_USER=####
#PUSHOVER_TOKEN=####
#SAVE CHANGES    TO THE .ENV FILES
#Run the test manually send a notification

In [109]:

load_dotenv()

True

In [110]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

In [111]:
#Test Pushover
import requests

def send_notification(message: str):
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [ ]:
#send_notification("Hello to myselft, from this AI engineering training")

###Step 3: Describe Pushover as an LLM tool

In [112]:
send_notification_function = {
    "name": "send_notification",
    "description": "sends a push notification",
    "parameters":{
        "type": "object",
        "properties":{
            "message":{
                "type": "string",
                "description": "The notification message to send to the user's device"
            }
        },
        "required": ["message"]
}
}

###STep 4: Add Pushover to the list of tool for the LLM

In [113]:
tools = [{"type":"function", "function":send_notification_function}]

###Steps 2b & 3b & 4b: Create new function, describe it, add it to the list of tools

In [114]:
import random
#simulates rolling a single six-sided die
def dice_roll():
    result = random.randint(1,6)
    return  result

#Describe fucntion for the LLM
roll_dice_function = {
      "name": "dice_roll",
      "description": "Simulates rolling a single six-sided die and returns the result. Use this when the user wants to roll a die for games, or random number generation.",
      "parameters":{
         "type": "object",
         "properties":{},
        "required": []
    }
}


#Add fucntion to list of tools of LLM
tools.append({f"type": "function", "function": roll_dice_function})


###Step 5: Calling the tool from an LLM

In [119]:
def handle_tool_call(tool_calls):
    tool_results = []

    for tool_call in tool_calls:
        function_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        #print(f" Calling function {function_name}")

        #Route to the appropriate function based on function_name
        if function_name == "send_notification":
             #tool_call = tool_calls[0] #assuming just one tool call
            send_notification(args["message"])
            content = f"Notification sent: {args['message']}"
        elif function_name == "dice_roll":
            content = f"Rolled: {dice_roll()}"
         #elif function_name == "insert_function_name_3":
        #    content = insert_function_name_3(args["messag"])

        else:
            content = f"Unknown functions: {function_name}"




           #Actually send the notification, i.e, call the tool
        
            #print(f"Sent notification: {args['message']}")

        tool_call_result = {
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call.id
    }

    tool_results.append(tool_call_result)
    
    return tool_results    #what to add to our "context"(about tool call), a dictionary.

In [120]:
client = OpenAI()
messages=[
      {"role": "user", "content": "Please do two things:\
            1)I'd like to roll 2 dice, and\
            2) Send me a notification saying hey there"}
    ]
     

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=messages,
    tools=tools
)

message = response.choices[0].message

#check if model wants to call a tool

if message.tool_calls:
    tool_result = handle_tool_call(message.tool_calls)
    messages.append(message)
    messages.extend(tool_result)
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages
    )
    message = response.choices[0].message

print(message.content)

BadRequestError: Error code: 400 - {'error': {'message': "An assistant message with 'tool_calls' must be followed by tool messages responding to each 'tool_call_id'. The following tool_call_ids did not have response messages: call_kOogkDKFYG1G7fEI2NUItd85, call_CXciJsUjnDXmNY9MosXz9wZ8", 'type': 'invalid_request_error', 'param': 'messages', 'code': None}}

In [102]:
print(message.content)

None
